# Random Forest & Logistic Regression Experiments

**Overview**

This notebook examines the performance of two models (Random Forest and Logistic Regression) under different strategies for handling class imbalance.  The following approaches are considered:

- Baseline (no handling)
- Class weight 
- SMOTE 
- SMOTEENN 

**To ensure a fair and consistent comparison, the following setup is maintained across all experiments:**

- All models share the same preprocessing pipeline to prevent data leakage.
- Stratified K-Fold cross-validation is applied to preserve class distribution across folds.
- A unified evaluation function is used to compute all metrics consistently.
- All results are stored in a shared table for cross-model and cross-strategy comparison.



## Setup

In [1]:
# Add project root to system path so we can import from src/
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent / "bank-deposit-prediction"
sys.path.append(str(PROJECT_ROOT))

# Standard libraries
import pandas as pd

# Modeling
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Imbalanced learning tools
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# Shared project modules
from src.shared import get_cv, get_preprocessing_steps
from src.evaluation import evaluate_model, save_results

## Data Loading
The dataset is loaded and split into features (X) and target variable (y).  
The target is encoded as binary values:
- 1 → "yes"
- 0 → "no"

In [2]:
data_path = PROJECT_ROOT / "data" / "raw" / "bank2.csv"

df = pd.read_csv(data_path, sep=';')

y = df['y'].map({'yes': 1, 'no': 0})
X = df.drop(columns=['y'])

## Shared components
The following components are used throughout the experiments:
- A centralized preprocessing pipeline to prevent data leakage
- Stratified K-Fold cross-validation, ensuring a fair evaluation

In [3]:
# Get shared cross-validation strategy
cv = get_cv()

## Random Forest



### Baseline Random Forest

We begin with the baseline model without applying any imbalance handling.

In [4]:
# Store experiment results
all_results = []

# Baseline Random Forest (no imbalance handling)
baseline_rf_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', RandomForestClassifier(random_state=42))
    ]
)

# Evaluate model using cross-validation
result_rf_baseline = evaluate_model(
    "RandomForest",
    "Baseline",
    baseline_rf_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_rf_baseline)

result_rf_baseline

{'Model': 'RandomForest',
 'Strategy': 'Baseline',
 'Accuracy': '0.8885 ± 0.0028',
 'Precision': '0.5940 ± 0.0983',
 'Recall': '0.1209 ± 0.0231',
 'F1': '0.1989 ± 0.0308',
 'PR-AUC': '0.3482 ± 0.0273',
 'ROC-AUC': '0.7333 ± 0.0184'}

### Class Weight Random Forest

In this experiment, class weights are introduced to give more importance to the minority class during training.

In [5]:
# Class weight Random Forest
cw_rf_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]
)

# Evaluate model using cross-validation
result_rf_cw = evaluate_model(
    "RandomForest",
    "ClassWeight",
    cw_rf_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_rf_cw)

result_rf_cw

{'Model': 'RandomForest',
 'Strategy': 'ClassWeight',
 'Accuracy': '0.8910 ± 0.0049',
 'Precision': '0.6479 ± 0.1295',
 'Recall': '0.1151 ± 0.0303',
 'F1': '0.1947 ± 0.0493',
 'PR-AUC': '0.3418 ± 0.0378',
 'ROC-AUC': '0.7384 ± 0.0211'}

### SMOTE Random Forest

In this experiment, SMOTE is applied to oversample the minority class and improve model sensitivity.

In [6]:
# SMOTE Random Forest
smote_rf_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smote', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(random_state=42))
    ]
)

# Evaluate model
result_rf_smote = evaluate_model(
    "RandomForest",
    "SMOTE",
    smote_rf_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_rf_smote)

result_rf_smote

{'Model': 'RandomForest',
 'Strategy': 'SMOTE',
 'Accuracy': '0.8848 ± 0.0051',
 'Precision': '0.5003 ± 0.0823',
 'Recall': '0.1516 ± 0.0294',
 'F1': '0.2320 ± 0.0416',
 'PR-AUC': '0.3234 ± 0.0372',
 'ROC-AUC': '0.7252 ± 0.0239'}

### SMOTEENN Random Forest

In this experiment, a hybrid approach (SMOTEENN) is applied to both oversample the minority class and clean noisy samples.

In [7]:
# SMOTEENN Random Forest
smoteenn_rf_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smoteenn', SMOTEENN(random_state=42)),
        ('model', RandomForestClassifier(random_state=42))
    ]
)

# Evaluate model
result_rf_smoteenn = evaluate_model(
    "RandomForest",
    "SMOTEENN",
    smoteenn_rf_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_rf_smoteenn)

result_rf_smoteenn

{'Model': 'RandomForest',
 'Strategy': 'SMOTEENN',
 'Accuracy': '0.8275 ± 0.0083',
 'Precision': '0.3171 ± 0.0158',
 'Recall': '0.4281 ± 0.0186',
 'F1': '0.3639 ± 0.0125',
 'PR-AUC': '0.3064 ± 0.0331',
 'ROC-AUC': '0.7207 ± 0.0273'}

## Logistic Regression

### Baseline Logistic Regression

We begin with the baseline model without applying any imbalance handling.

In [8]:
# Baseline Logistic Regression
baseline_lr_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', LogisticRegression(max_iter=5000, random_state=42))
    ]
)

# Evaluate model
result_lr_baseline = evaluate_model(
    "LogisticRegression",
    "Baseline",
    baseline_lr_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_lr_baseline)

result_lr_baseline

{'Model': 'LogisticRegression',
 'Strategy': 'Baseline',
 'Accuracy': '0.8923 ± 0.0037',
 'Precision': '0.6605 ± 0.1145',
 'Recall': '0.1439 ± 0.0290',
 'F1': '0.2342 ± 0.0399',
 'PR-AUC': '0.3482 ± 0.0375',
 'ROC-AUC': '0.7247 ± 0.0143'}

### Class Weight (Imbalance Handling)

- Adjusts the model to give more importance to the minority class
- Helps improve recall for the positive class without changing the data

In [9]:
# Logistic Regression with class weight

lr_cw_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=5000,
            random_state=42
        ))
    ]
)

# Evaluate model
result_lr_cw = evaluate_model(
    "LogisticRegression",
    "ClassWeight",
    lr_cw_pipeline,
    X,
    y,
    cv
)

# Save results
all_results.append(result_lr_cw)

# Display results
result_lr_cw

{'Model': 'LogisticRegression',
 'Strategy': 'ClassWeight',
 'Accuracy': '0.7125 ± 0.0128',
 'Precision': '0.2188 ± 0.0073',
 'Recall': '0.5815 ± 0.0427',
 'F1': '0.3176 ± 0.0122',
 'PR-AUC': '0.3407 ± 0.0344',
 'ROC-AUC': '0.7238 ± 0.0130'}

### SMOTE Logistic Regression

SMOTE is applied to oversample the minority class and improve the model’s ability to detect positive cases.

In [10]:
# SMOTE Logistic Regression
lr_smote_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smote', SMOTE(random_state=42)),
        ('model', LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]
)

# Evaluate model
result_lr_smote = evaluate_model(
    "LogisticRegression",
    "SMOTE",
    lr_smote_pipeline,
    X,
    y,
    cv
)

# Save results
all_results.append(result_lr_smote)

# Display results
result_lr_smote

{'Model': 'LogisticRegression',
 'Strategy': 'SMOTE',
 'Accuracy': '0.7211 ± 0.0124',
 'Precision': '0.2252 ± 0.0059',
 'Recall': '0.5815 ± 0.0346',
 'F1': '0.3244 ± 0.0084',
 'PR-AUC': '0.3350 ± 0.0388',
 'ROC-AUC': '0.7166 ± 0.0178'}

### SMOTEENN Logistic Regression

A hybrid approach combining SMOTE and ENN is applied to both oversample the minority class and remove noisy observations.

In [11]:
# SMOTEENN Logistic Regression
lr_smoteenn_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smoteenn', SMOTEENN(random_state=42)),
        ('model', LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]
)

# Evaluate model
result_lr_smoteenn = evaluate_model(
    "LogisticRegression",
    "SMOTEENN",
    lr_smoteenn_pipeline,
    X,
    y,
    cv
)

# Save results
all_results.append(result_lr_smoteenn)

# Display results
result_lr_smoteenn

{'Model': 'LogisticRegression',
 'Strategy': 'SMOTEENN',
 'Accuracy': '0.5417 ± 0.0202',
 'Precision': '0.1680 ± 0.0072',
 'Recall': '0.7524 ± 0.0428',
 'F1': '0.2745 ± 0.0115',
 'PR-AUC': '0.3284 ± 0.0279',
 'ROC-AUC': '0.7205 ± 0.0172'}

In [12]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Save to shared results table
save_results(results_df)

In [13]:
results_path = PROJECT_ROOT / "results" / "experiment_results.csv"
results = pd.read_csv(results_path)

results

,Model,Strategy,Accuracy,Precision,Recall,F1,PR-AUC,ROC-AUC
0,XGBoost,Baseline,0.8830 ± 0.0048,0.4765 ± 0.0576,0.1784 ± 0.0353,0.2590 ± 0.0447,0.3037 ± 0.0329,0.7020 ± 0.0049
1,XGBoost,ClassWeight,0.8498 ± 0.0070,0.3300 ± 0.0308,0.2936 ± 0.0394,0.3099 ± 0.0313,0.2791 ± 0.0381,0.6806 ± 0.0194
2,XGBoost,SMOTE,0.8834 ± 0.0023,0.4829 ± 0.0311,0.1765 ± 0.0372,0.2567 ± 0.0431,0.3178 ± 0.0177,0.7172 ± 0.0150
3,XGBoost,SMOTEENN,0.8290 ± 0.0041,0.3266 ± 0.0061,0.4549 ± 0.0160,0.3800 ± 0.0061,0.3323 ± 0.0215,0.7270 ± 0.0142
4,RandomForest,Baseline,0.8885 ± 0.0028,0.5940 ± 0.0983,0.1209 ± 0.0231,0.1989 ± 0.0308,0.3482 ± 0.0273,0.7333 ± 0.0184
5,RandomForest,ClassWeight,0.8910 ± 0.0049,0.6479 ± 0.1295,0.1151 ± 0.0303,0.1947 ± 0.0493,0.3418 ± 0.0378,0.7384 ± 0.0211
6,RandomForest,SMOTE,0.8848 ± 0.0051,0.5003 ± 0.0823,0.1516 ± 0.0294,0.2320 ± 0.0416,0.3234 ± 0.0372,0.7252 ± 0.0239
7,RandomForest,SMOTEENN,0.8275 ± 0.0083,0.3171 ± 0.0158,0.4281 ± 0.0186,0.3639 ± 0.0125,0.3064 ± 0.0331,0.7207 ± 0.0273
8,LogisticRegression,Baseline,0.8923 ± 0.0037,0.6605 ± 0.1145,0.1439 ± 0.0290,0.2342 ± 0.0399,0.3482 ± 0.0375,0.7247 ± 0.0143
9,LogisticRegression,ClassWeight,0.7125 ± 0.0128,0.2188 ± 0.0073,0.5815 ± 0.0427,0.3176 ± 0.0122,0.3407 ± 0.0344,0.7238 ± 0.0130
